# 07 — Ministry of Agriculture Prices | Silver Transformation

One row per **channel × market × product × unit × year × month × week of month**, price in DOP (`silver.ma_agri_prices_weekly`).

| Rule | Treatment |
|---|---|
| Snapshot | Each report carries the history since August 2017, so Silver uses the newest Excel snapshot of each channel. Excel is authoritative (the CSV rounds prices); an observational check compares both formats cell by cell. |
| Layout | Retail rows must have 14 cells and wholesale rows 8. Rows with another shape (7 in the current files) are **rejected**, never guessed. Repeated header rows are skipped and counted. |
| Retail markets | The seven retail price columns are unpivoted into `market`: Mercado Nuevo, Mercado Conaprope, Mercado Los Mina, Mercado Villa Consuelo, Mercado Cristo Rey, Mercadom, Supermercado. Wholesale has one price without a location (`market = 'Mayorista'`). These are markets of Gran Santo Domingo, **not regions or provinces**. |
| Week, month, year | Controlled mappings with audit: week labels with typos (`Sengunda`, `Cuanta`, `Priemera`, `Tercero`…) map to weeks 1–5; `Juio` → July; stray characters are removed (`` `mayo``, ``2025` ``). Anything else (`3333`, blank) is rejected. |
| Price | Numeric and > 0 → price. `0`, `N/D`, blank, `-`, `.` → **not quoted** (counted, not stored). Excel errors (`#REF!`, `#DIV/0!`) and other text → rejected. |
| Rubro (product group) | Controlled dictionary: spelling variants (`Lateo`, `Lateos`, `Celeares`, `Pescuario`, `Oleginosas`…) map to 10 groups. Unknown values become `Sin clasificar` and are reported. |
| Product and unit labels | Labels equal ignoring case and spaces get one canonical label (most frequent spelling); unit synonyms `Lib`/`lb`, `Und`/`Unidad`, `Doc`/`Docena` are unified. Originals kept in `source_product`, `source_unit`. Products with different words are never merged. |
| Duplicates | One row per business key; the **later row in the file** is kept, earlier copies are rejected. |

Target: `silver.ma_agri_prices_weekly` · full refresh · **Status:** Development.

In [0]:
import uuid, json
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "rd_economic_intelligence"
SOURCE = f"{CATALOG}.bronze.ma_agri_prices_raw"
TARGET = f"{CATALOG}.silver.ma_agri_prices_weekly"
REJECTED = f"{CATALOG}.quality.dq_rejected_records"
RESULTS = f"{CATALOG}.quality.dq_rule_results"
RUN_ID, RUN_TS = str(uuid.uuid4()), datetime.now(timezone.utc)
MARKETS = ["Mercado Nuevo", "Mercado Conaprope", "Mercado Los Mina", "Mercado Villa Consuelo", "Mercado Cristo Rey", "Mercadom", "Supermercado"]
WEEKS = {"primera": 1, "primero": 1, "priemera": 1, "segunda": 2, "segundo": 2, "sengunda": 2, "tercera": 3, "tercero": 3,
         "cuarta": 4, "cuarto": 4, "cuanta": 4, "quinta": 5, "quinto": 5}
MONTHS = {"enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6, "julio": 7, "juio": 7, "agosto": 8,
          "septiembre": 9, "setiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12}
RUBROS = {"frutas": "Frutas", "legumbreshortalizas": "Legumbres y hortalizas", "legumbresyhortalizas": "Legumbres y hortalizas",
          "pecuarios": "Pecuarios", "pecuario": "Pecuarios", "pescuario": "Pecuarios",
          "raicestuberculos": "Raíces y tubérculos", "raicesytuberculos": "Raíces y tubérculos", "musaceas": "Musáceas",
          "leguminosassecas": "Leguminosas secas", "cereales": "Cereales", "cereal": "Cereales", "celeares": "Cereales", "celeales": "Cereales",
          "avicolas": "Avícolas", "avicolass": "Avícolas", "oleaginosas": "Oleaginosas", "oleginosas": "Oleaginosas",
          "lacteos": "Lácteos", "lateo": "Lácteos", "lateos": "Lácteos"}
UNITS = {"lb": "lb", "lib": "lb", "libra": "lb", "libras": "lb", "und": "Unidad", "unidad": "Unidad", "ud": "Unidad", "unid": "Unidad",
         "doc": "Docena", "docena": "Docena", "paq": "Paquete", "paquete": "Paquete"}
NOT_QUOTED = ["", "0", "N/D", "ND", "-", "."]

def as_map(d):
    return F.create_map(*[F.lit(x) for kv in d.items() for x in kv])
def letters_key(col):   # lower case, accents removed, only letters
    return F.regexp_replace(F.translate(F.lower(F.trim(col)), "áéíóúüñ", "aeiouun"), "[^a-z]", "")

In [0]:
# Newest Excel snapshot per channel
src = spark.table(SOURCE)
excel = src.filter("_source_format = 'excel'")
newest = excel.groupBy("_channel").agg(F.max_by("_source_path", "_source_modified_ts").alias("_source_path"))
rows = (excel.join(newest, ["_channel", "_source_path"])
        .withColumn("c", F.from_json("row_json", "array<string>")))
bronze_rows = rows.count()

is_header = F.coalesce(F.lower(F.trim(F.col("c")[0])) == "productos", F.lit(False))
layout_ok = ((F.col("_channel") == "retail") & (F.col("cell_count") == 14)) | ((F.col("_channel") == "wholesale") & (F.col("cell_count") == 8))
headers = rows.filter(is_header)
bad_layout = rows.filter(~is_header & ~layout_ok)
data = rows.filter(~is_header & layout_ok)

retail = F.col("_channel") == "retail"
data = (data
    .withColumn("source_product", F.trim(F.col("c")[0]))
    .withColumn("source_rubro", F.trim(F.col("c")[1]))
    .withColumn("source_unit", F.trim(F.col("c")[3]))
    .withColumn("wk_raw", F.when(retail, F.col("c")[11]).otherwise(F.col("c")[5]))
    .withColumn("mo_raw", F.when(retail, F.col("c")[12]).otherwise(F.col("c")[6]))
    .withColumn("yr_raw", F.when(retail, F.col("c")[13]).otherwise(F.col("c")[7]))
    .withColumn("week_of_month", as_map(WEEKS)[letters_key("wk_raw")])
    .withColumn("month", as_map(MONTHS)[letters_key("mo_raw")])
    .withColumn("_yd", F.regexp_replace("yr_raw", "[^0-9]", ""))
    .withColumn("year", F.when((F.length("_yd") == 4) & F.col("_yd").cast("int").between(2017, RUN_TS.year), F.col("_yd").cast("int")))
    .withColumn("period_cleaned", (F.trim("yr_raw") != F.col("_yd")) | (letters_key("mo_raw") == "juio") |
                (F.lower(F.trim("mo_raw")) != letters_key("mo_raw")) | (F.lower(F.trim("wk_raw")) != letters_key("wk_raw"))))
period_ok = F.col("week_of_month").isNotNull() & F.col("month").isNotNull() & F.col("year").isNotNull()
bad_period = data.filter(~period_ok)
data = data.filter(period_ok)

# Unpivot: retail 7 market columns, wholesale 1 price
market_cells = F.array(*[F.struct(F.lit(m).alias("market"), F.col("c")[4 + i].alias("price_raw")) for i, m in enumerate(MARKETS)])
cells = (data.withColumn("cell", F.when(retail, market_cells)
                         .otherwise(F.array(F.struct(F.lit("Mayorista").alias("market"), F.col("c")[4].alias("price_raw")))))
         .withColumn("cell", F.explode("cell"))
         .select("*", F.col("cell.market").alias("market"), F.trim(F.col("cell.price_raw")).alias("price_raw")).drop("cell"))
num = F.expr("try_cast(regexp_replace(price_raw, ',', '') AS double)")
cells = cells.withColumn("_num", num).withColumn("price_status",
    F.when(F.coalesce(F.col("price_raw"), F.lit("")).isin(NOT_QUOTED) | (F.col("_num") == 0), "not_quoted")
     .when(F.col("_num") > 0, "ok").otherwise("error"))

not_quoted = cells.filter("price_status = 'not_quoted'").count()
price_errors = cells.filter("price_status = 'error'")
priced = cells.filter("price_status = 'ok'").withColumn("price_dop", F.round("_num", 2).cast("decimal(12,2)"))

# Controlled dictionaries and canonical labels
priced = (priced.withColumn("rubro", F.coalesce(as_map(RUBROS)[letters_key("source_rubro")], F.lit("Sin clasificar")))
                .withColumn("_ukey", F.lower(F.regexp_replace("source_unit", r"\s+", "")))
                .withColumn("_unit", F.coalesce(as_map(UNITS)[F.col("_ukey")], F.col("source_unit")))
                .withColumn("_pkey", F.lower(F.regexp_replace("source_product", r"\s+", ""))))
def canonical(df, key, label, out):
    freq = df.groupBy(key, label).count()
    best = freq.withColumn("_r", F.row_number().over(Window.partitionBy(key).orderBy(F.desc("count"), F.desc(label)))).filter("_r = 1").select(key, F.col(label).alias(out))
    return df.join(best, key)
priced = canonical(priced, "_pkey", "source_product", "product")
priced = priced.withColumn("_ukey2", F.lower(F.regexp_replace("_unit", r"\s+", "")))
priced = canonical(priced, "_ukey2", "_unit", "unit")

# Duplicates: later row in the file wins
key = ["_channel", "market", "product", "unit", "year", "month", "week_of_month"]
w = Window.partitionBy(*key).orderBy(F.desc("_source_row_number"))
ranked = priced.withColumn("_rn", F.row_number().over(w))
dups = ranked.filter("_rn > 1")
silver = (ranked.filter("_rn = 1")
          .select(F.col("_channel").alias("channel"), "market", "rubro", "product", "unit", "year", "month", "week_of_month",
                  F.make_date("year", "month", F.lit(1)).alias("reference_month"), "price_dop",
                  "source_product", "source_rubro", "source_unit", "period_cleaned",
                  "_source_file", "_source_row_number", F.col("_batch_id").alias("_bronze_batch_id"))
          .withColumn("_silver_run_id", F.lit(RUN_ID)).withColumn("_silver_processed_ts", F.lit(RUN_TS).cast("timestamp")))

In [0]:
# Rule results and rejected records (reconciliation is checked before anything is written)
n_silver = silver.count()
n_cells = cells.count(); n_err = price_errors.count(); n_dup = dups.count()
n_hdr, n_layout, n_period = headers.count(), bad_layout.count(), bad_period.count()
n_unclassified = silver.filter("rubro = 'Sin clasificar'").count()
n_cleaned = data.filter("period_cleaned").count()

# Cross-format reconciliation (observational): newest CSV vs newest Excel, same row and cell, prices within 0.01
csv = src.filter("_source_format = 'csv'")
newest_csv = csv.groupBy("_channel").agg(F.max_by("_source_path", "_source_modified_ts").alias("_source_path"))
def price_cells(df):
    d = df.withColumn("c", F.from_json("row_json", "array<string>")).filter(layout_ok)
    d = d.withColumn("pc", F.when(retail, F.slice("c", 5, 7)).otherwise(F.slice("c", 5, 1)))
    return d.select("_channel", "_source_row_number", F.posexplode("pc").alias("pos", "p")) \
            .withColumn("v", F.expr("try_cast(regexp_replace(trim(p), ',', '') AS double)"))
xc = price_cells(excel.join(newest, ["_channel", "_source_path"])).alias("x")
cc = price_cells(csv.join(newest_csv, ["_channel", "_source_path"])).alias("y")
cmp = xc.join(cc, ["_channel", "_source_row_number", "pos"], "full_outer")
cmp_tested = cmp.filter("x.v IS NOT NULL OR y.v IS NOT NULL").count()
cmp_diff = cmp.filter("x.v IS NULL OR y.v IS NULL OR abs(x.v - y.v) > 0.01").filter("NOT (coalesce(x.v,0) = 0 AND coalesce(y.v,0) = 0)").count()

checks = []
def add(name, rtype, tested, failed):
    status = "PASS" if failed == 0 else ("WARN" if rtype == "observational" else "REJECTED")
    checks.append((RUN_ID, TARGET, name, rtype, int(tested), int(failed), status, RUN_TS))
add("row_layout_14_retail_8_wholesale", "enforced", bronze_rows - n_hdr, n_layout)
add("valid_week_month_year", "enforced", bronze_rows - n_hdr - n_layout, n_period)
add("price_numeric_or_not_quoted", "enforced", n_cells, n_err)
add("unique_channel_market_product_unit_week", "enforced", n_cells - n_err - not_quoted, n_dup)
add("price_not_quoted_zero_or_nd", "observational", n_cells, not_quoted)
add("period_label_cleaned_by_mapping", "observational", bronze_rows - n_hdr - n_layout, n_cleaned)
add("rubro_unclassified", "observational", n_silver, n_unclassified)
add("repeated_header_rows_skipped", "observational", bronze_rows, max(n_hdr - 1, 0))
add("excel_csv_price_cells_match", "observational", cmp_tested, cmp_diff)
# Reconciliation (enforced): rows and price cells are fully accounted for
rows_ok = bronze_rows == n_hdr + n_layout + n_period + data.count()
cells_ok = n_cells == n_silver + not_quoted + n_err + n_dup
add("reconciliation_rows_and_cells", "enforced", n_cells, 0 if (rows_ok and cells_ok) else 1)
for c in checks: print(f"  {c[6]:<8} {c[2]:<44} failed={c[5]:,}/{c[4]:,}")
assert rows_ok and cells_ok, (f"Reconciliation failed: rows {bronze_rows} vs {n_hdr}+{n_layout}+{n_period}+{data.count()}; "
                              f"cells {n_cells} vs {n_silver}+{not_quoted}+{n_err}+{n_dup}")
silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TARGET)

def rej(df, rule, keycol):
    return df.select(F.lit(RUN_ID).alias("run_id"), F.lit(TARGET).alias("target_table"), F.lit(rule).alias("rule_name"),
                     keycol.alias("record_key"), F.col("row_json").alias("record_json"), F.col("_batch_id").alias("source_batch_id"),
                     F.lit(RUN_TS).cast("timestamp").alias("detected_ts"))
rk = F.concat_ws("#", "_channel", "_source_file", F.col("_source_row_number").cast("string"))
(rej(bad_layout, "row_layout_14_retail_8_wholesale", rk)
 .unionByName(rej(bad_period, "valid_week_month_year", rk))
 .unionByName(rej(price_errors, "price_numeric_or_not_quoted", F.concat_ws("#", rk, "market")))
 .unionByName(rej(dups, "unique_channel_market_product_unit_week", F.concat_ws("#", rk, "market")))
 .write.mode("append").saveAsTable(REJECTED))
spark.createDataFrame(checks, "run_id string, target_table string, rule_name string, rule_type string, records_tested bigint, records_failed bigint, status string, run_ts timestamp").write.mode("append").saveAsTable(RESULTS)
spark.sql(f"COMMENT ON TABLE {TARGET} IS 'Silver: Ministry of Agriculture weekly market prices (DOP) by channel (retail/wholesale), market, product group (rubro), product and unit, since August 2017. Markets are in Gran Santo Domingo; not regions.'")

s = spark.sql(f"SELECT channel, COUNT(*) n, COUNT(DISTINCT product) p, COUNT(DISTINCT rubro) r, MIN(reference_month) a, MAX(reference_month) z FROM {TARGET} GROUP BY channel").collect()
dbutils.notebook.exit(json.dumps({"status": "SUCCESS", "bronze_rows": bronze_rows, "price_cells": n_cells, "silver_rows": n_silver,
    "by_channel": {r["channel"]: {"rows": r["n"], "products": r["p"], "rubros": r["r"], "months": [str(r["a"]), str(r["z"])]} for r in s},
    "not_quoted": not_quoted, "rejected": {"layout": n_layout, "period": n_period, "price_error": n_err, "duplicate": n_dup},
    "checks": {c[2]: f"{c[6]} ({c[5]}/{c[4]})" for c in checks}}, ensure_ascii=False))